# Step 20.5 — Resume Verification

本节是大 Step 20 最后一节，范围严格对应 roadmap：**small deterministic test** 与 **architecture/config mismatch detection**。20.4 最终提交 `5af3826cb22df76338e2768500e9c2d1f05e891f` 已通过核心静态验收（常规2/2 + 特批1次）；20.3累计1/2，20.2累计1/2，19.5为2/2。这些是材料验收，不是运行结果或学员掌握证明。

本节初始生成0/2。所有代码单元未执行、无输出；不加载真实checkpoint、不训练、不读取语料。完成本节后，Step 20 材料范围齐备；这不代表恢复流程已经运行验证。

## 1. Verification 的目的与边界

**Resume verification（恢复验证）**是在未来工作副本中，确认恢复后的对象与保存前记录一致。它不是训练，也不证明长期训练质量。

本节使用人工构造的固定 token ID 小输入，不读取语料。主要检查：

1. architecture 与 tokenizer 配置是否匹配；
2. 同一输入下恢复前后的 logits 是否一致；
3. 模型参数树与 Adam 状态树是否一致；
4. Adam step 是否与 metadata 的 completed_updates 一致；
5. 故意不匹配的配置是否会被拒绝。

**Deterministic test（确定性测试）**只在固定输入、相同参数、相同执行条件与容差范围内比较结果。跨设备、MLX版本或不同计算路径不保证逐位相同。

In [ ]:
import json
from pathlib import Path

import mlx.core as mx
import mlx.nn as nn
import mlx.optimizers as optim
from mlx.utils import tree_flatten

# Tiny deterministic verification configuration; not a training configuration.
VERIFY_CONFIG = {
    "vocab_size": 11,
    "dims": 16,
    "num_heads": 4,
    "num_blocks": 1,
    "max_seq_len": 8,
}
VERIFY_ADAM_CONFIG = {
    "learning_rate": 3e-4,
    "betas": [0.9, 0.999],
    "eps": 1e-8,
    "bias_correction": True,
}

def require(condition, message):
    if not condition:
        raise AssertionError(message)

def assert_architecture_compatible(saved_config, current_config):
    keys = ("vocab_size", "dims", "num_heads", "num_blocks", "max_seq_len")
    mismatches = {
        key: (saved_config.get(key), current_config.get(key))
        for key in keys if saved_config.get(key) != current_config.get(key)
    }
    if mismatches:
        raise ValueError(f"Architecture/config mismatch: {mismatches}")
    if current_config["dims"] % current_config["num_heads"] != 0:
        raise ValueError("dims must be divisible by num_heads.")
    return True

## 2. 完整 MiniGPT 定义与小型测试输入

沿用 Step 15.4 的模型实现，但本节使用独立的小型 verification config，目的是缩小未来测试的输入规模，而不是替换课程模型或训练基线。

固定输入 (X) 的 shape 为 ((B,T)=(2,5))，token ID 取值范围为 ([0,V-1])。模型输出 logits 的预期 shape 是 ((2,5,11))。

In [ ]:
import math
import mlx.core as mx
import mlx.nn as nn


class CausalMultiHeadAttention(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()

        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        self.dims = dims
        self.num_heads = num_heads
        self.head_dim = dims // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)  # Python float, not trainable

        self.qkv_proj = nn.Linear(
            input_dims=dims,
            output_dims=3 * dims,
            bias=False,
        )
        self.c_proj = nn.Linear(
            input_dims=dims,
            output_dims=dims,
            bias=False,
        )

    def __call__(self, x: mx.array) -> mx.array:
        if len(x.shape) != 3:
            raise ValueError('attention input must have shape (B,T,C)')

        B, T, C = x.shape
        if C != self.dims:
            raise ValueError('attention input width must equal dims')

        H = self.num_heads
        D = self.head_dim

        qkv = self.qkv_proj(x)  # (B,T,3C)
        q, k, v = mx.split(qkv, indices_or_sections=3, axis=-1)

        q = mx.reshape(q, (B, T, H, D))
        k = mx.reshape(k, (B, T, H, D))
        v = mx.reshape(v, (B, T, H, D))

        q = mx.swapaxes(q, 1, 2)  # (B,H,T,D)
        k = mx.swapaxes(k, 1, 2)
        v = mx.swapaxes(v, 1, 2)

        scores = mx.matmul(
            q,
            mx.swapaxes(k, -1, -2),
        ) * self.scale  # (B,H,T,T)

        # IMPORTANT: structural mask is local to forward.
        # It is NOT stored as self.causal_mask.
        future = mx.triu(mx.ones((T, T)), k=1) > 0
        mask = mx.where(
            future,
            mx.array(float('-inf'), dtype=scores.dtype),
            mx.array(0.0, dtype=scores.dtype),
        )

        weights = mx.softmax(scores + mask, axis=-1)
        head_out = mx.matmul(weights, v)  # (B,H,T,D)

        token_major = mx.swapaxes(head_out, 1, 2)  # (B,T,H,D)
        merged = mx.reshape(token_major, (B, T, C))
        return self.c_proj(merged)  # (B,T,C)


class FeedForwardMLP(nn.Module):
    def __init__(self, dims: int):
        super().__init__()
        self.fc1 = nn.Linear(dims, 4 * dims, bias=True)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * dims, dims, bias=True)

    def __call__(self, x: mx.array) -> mx.array:
        return self.fc2(self.act(self.fc1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()
        self.norm1 = nn.RMSNorm(dims=dims)
        self.attn = CausalMultiHeadAttention(dims=dims, num_heads=num_heads)
        self.norm2 = nn.RMSNorm(dims=dims)
        self.mlp = FeedForwardMLP(dims=dims)

    def __call__(self, x: mx.array) -> mx.array:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


# Definitions only. This master notebook does not execute them.


In [ ]:
class MiniGPT(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        dims: int,
        num_heads: int,
        num_blocks: int,
        max_seq_len: int,
    ):
        super().__init__()

        if vocab_size <= 0:
            raise ValueError('vocab_size must be positive')
        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if num_blocks <= 0:
            raise ValueError('num_blocks must be positive')
        if max_seq_len <= 0:
            raise ValueError('max_seq_len must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        # Python metadata: these are not trainable mx.array parameters.
        self.vocab_size = vocab_size
        self.dims = dims
        self.max_seq_len = max_seq_len

        # Learned lookup tables.
        self.token_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            dims=dims,
        )
        self.position_embedding = nn.Embedding(
            num_embeddings=max_seq_len,
            dims=dims,
        )

        # A list of distinct Module instances; MLX recursively tracks them.
        self.blocks = [
            TransformerBlock(
                dims=dims,
                num_heads=num_heads,
            )
            for _ in range(num_blocks)
        ]

        self.final_norm = nn.RMSNorm(dims=dims)
        self.lm_head = nn.Linear(
            input_dims=dims,
            output_dims=vocab_size,
            bias=False,
        )

    def __call__(self, token_ids: mx.array) -> mx.array:
        if len(token_ids.shape) != 2:
            raise ValueError('token_ids must have shape (B,T)')

        B, T = token_ids.shape

        if T < 1:
            raise ValueError('sequence length T must be at least 1')
        if T > self.max_seq_len:
            raise ValueError(
                f'sequence length {T} exceeds max_seq_len {self.max_seq_len}'
            )

        # Forward-local structural indices: not Module parameters.
        position_ids = mx.arange(T)  # (T,)

        token_features = self.token_embedding(token_ids)        # (B,T,C)
        position_features = self.position_embedding(position_ids)  # (T,C)
        x = token_features + position_features                  # (B,T,C)

        for block in self.blocks:
            x = block(x)  # invariant: (B,T,C)

        x = self.final_norm(x)  # (B,T,C)
        logits = self.lm_head(x)  # (B,T,V)

        expected_shape = (B, T, self.vocab_size)
        if logits.shape != expected_shape:
            raise RuntimeError(
                f'logits shape {logits.shape} != expected {expected_shape}'
            )

        return logits


# Definition only. No MiniGPT forward is executed in this master notebook.


In [ ]:
def make_verification_input():
    # Fixed IDs, no tokenizer or corpus access.
    return mx.array(
        [[1, 2, 3, 4, 5],
         [5, 4, 3, 2, 1]],
        dtype=mx.int32,
    )

def make_verification_model(config):
    model = MiniGPT(
        vocab_size=config["vocab_size"],
        dims=config["dims"],
        num_heads=config["num_heads"],
        num_blocks=config["num_blocks"],
        max_seq_len=config["max_seq_len"],
    )
    return model

# Future work-copy setup only; not executed in this published master.
# mx.random.seed(2026)
# reference_model = make_verification_model(VERIFY_CONFIG)
# X_verify = make_verification_input()
# reference_logits = reference_model(X_verify)
# mx.eval(reference_model.parameters(), reference_logits)

## 3. 比较模型参数与 logits

**Logits comparison（输出比较）**检查同一输入在保存前模型与恢复后模型上的输出是否接近。由于浮点计算可能存在微小差异，使用 `mx.allclose` 而不是直接要求每个浮点数完全相等。

**Parameter tree comparison（参数树比较）**逐路径比较参数结构、shape、dtype 与数值。参数一致是 logits 一致的重要前提，但 logits 测试还可以捕获前向路径或配置连接错误。

In [ ]:
def compare_array_trees(reference_tree, restored_tree, *, rtol=1e-5, atol=1e-6):
    reference = tree_flatten(reference_tree)
    restored = tree_flatten(restored_tree)
    if [str(path) for path, _ in reference] != [str(path) for path, _ in restored]:
        raise AssertionError("Tree paths/order differ.")
    for (path_a, a), (path_b, b) in zip(reference, restored):
        if not isinstance(a, mx.array) or not isinstance(b, mx.array):
            if type(a) is not type(b) or a != b:
                raise AssertionError(f"Non-array leaf differs at {path_a}.")
            continue
        if tuple(a.shape) != tuple(b.shape) or a.dtype != b.dtype:
            raise AssertionError(f"Shape/dtype differs at {path_a}.")
        mx.eval(a, b)
        if not bool(mx.allclose(a, b, rtol=rtol, atol=atol).item()):
            raise AssertionError(f"Array values differ at {path_a}.")
    return True

def compare_logits(reference_model, restored_model, X, *, rtol=1e-5, atol=1e-6):
    reference_logits = reference_model(X)
    restored_logits = restored_model(X)
    mx.eval(reference_logits, restored_logits)
    if reference_logits.shape != restored_logits.shape:
        raise AssertionError(
            f"Logit shape mismatch: {reference_logits.shape} vs {restored_logits.shape}"
        )
    if not bool(mx.allclose(reference_logits, restored_logits, rtol=rtol, atol=atol).item()):
        raise AssertionError("Same-input logits differ beyond tolerance.")
    return True

## 4. Adam 状态与更新进度核对

模型权重相同，不代表 Adam 历史相同。本节沿用 20.2 的状态树结构，逐路径比较 moments、step 和当前 learning_rate 等数组叶子。之后再核对 Adam step 与 metadata 的 completed_updates。

Step 是零维 MLX 数组；先显式求值，再读取其整数值。

In [ ]:
def verify_optimizer_state(reference_optimizer, restored_optimizer):
    return compare_array_trees(reference_optimizer.state, restored_optimizer.state)

def read_adam_step(optimizer):
    flattened = tree_flatten(optimizer.state)
    matches = [(path, value) for path, value in flattened if str(path).endswith("step")]
    if len(matches) != 1:
        raise ValueError(f"Expected one Adam step leaf, found {len(matches)}.")
    path, value = matches[0]
    if not isinstance(value, mx.array) or value.shape != ():
        raise TypeError(f"Adam step must be a zero-dimensional MLX array, got {type(value)} / {getattr(value, 'shape', None)}.")
    mx.eval(value)
    return int(value.item())

def verify_progress(optimizer, metadata):
    expected = metadata.get("training_progress", {}).get("completed_updates")
    if type(expected) is not int or expected < 0:
        raise ValueError("Metadata completed_updates must be a real non-negative integer.")
    actual = read_adam_step(optimizer)
    if actual != expected:
        raise AssertionError(f"Adam step {actual} != metadata completed_updates {expected}.")
    return True

## 5. 架构与配置 mismatch 检测

在真正加载权重前，先比较保存的 architecture 与当前准备构造的配置。下面的负向测试只验证纯 Python 配置检查：故意改变 width 或 vocab_size 应触发明确异常。它不需要模型、checkpoint 或语料。

未来实际恢复时还应比较 tokenizer fingerprint、Adam 配置、artifact SHA-256、optimizer state tree 路径与 step。

In [ ]:
def expect_architecture_mismatch(saved_config, candidate_config):
    try:
        assert_architecture_compatible(saved_config, candidate_config)
    except ValueError as exc:
        return str(exc)
    raise AssertionError("Expected an architecture/config mismatch, but none was detected.")

# Illustrative negative cases; calls are examples for the future learner, not run here.
# wrong_width = dict(VERIFY_CONFIG, dims=32)
# expect_architecture_mismatch(VERIFY_CONFIG, wrong_width)
# wrong_vocab = dict(VERIFY_CONFIG, vocab_size=12)
# expect_architecture_mismatch(VERIFY_CONFIG, wrong_vocab)

## 6. 保存前后验证的未来执行顺序

未来工作副本按以下顺序组织验证：

1. 从同一 run 的保存前对象取得 reference model/optimizer，或在保存阶段额外记录必要的比较快照。
2. 记录 architecture、tokenizer fingerprint、completed_updates 与 artifact hashes。
3. 使用 20.4 的恢复流程重建 restored model/optimizer。
4. 先比较参数树，再用固定 (X) 比较 logits。
5. 比较 optimizer state tree 与 Adam step。
6. 用故意不匹配的配置验证 mismatch 检查能否拒绝错误输入。

如果没有保存前的 reference logits/参数快照或可重建的 reference 对象，就不能事后声称完成了保存前后数值一致性比较。

## 7. 验证结论应该如何表述？

**PASS（通过）**只能表示具体执行过的检查在指定环境与容差下通过。它不代表训练质量、泛化能力或跨设备 bitwise reproducibility。

建议未来记录：

- MLX/Python版本与设备；
- architecture 与 tokenizer fingerprint；
- reference/restored artifact SHA-256；
- 输入 (X) 的 shape 与固定 token IDs；
- logits 的最大绝对差与 allclose 容差；
- 参数树、optimizer state tree 的比较结果；
- Adam step 与 completed_updates；
- mismatch negative test 是否按预期拒绝。

没有执行的项目必须标记为 **not run / not measured**，不能预填 PASS。

## Step 20 完成边界与累计计数

本节是 roadmap 中 Step 20 的最后一节。20.1–20.5 的教学材料范围现已全部准备；这表示课程材料齐备，不代表 checkpoint 已实际保存/恢复，也不代表恢复验证已执行通过。

20.5 初始生成 0/2；20.4 常规修正 2/2 + 特批额外 1 次；20.3 为 1/2；20.2 为 1/2；20.1 为 0/2；19.5 为 2/2；19.2/19.3 各 1/2，其余历史计数保留。

所有代码单元未执行、无输出。本轮没有运行 Notebook、训练、验证、测试、实验、采样、测量、工作副本或真实 checkpoint 操作，也没有读取/下载语料。完成 Step 20.5 后停止，不进入 Step 21。